In [9]:
import pandas as pd
import numpy as np


import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import classification_report, roc_auc_score

# Load both versions
df = pd.read_csv("../data/processed/features_2023.csv")
df_logreg = pd.read_csv("../data/processed/features_2023_scaled.csv")


print(f"XGBoost dataset: {df.shape}")
print(f"Logistic Regression dataset: {df_logreg.shape}")

XGBoost dataset: (11089, 34)
Logistic Regression dataset: (11089, 27)


In [13]:
# Filter to only the laps where a pit stop actually happened 

pit_laps = df[df["Pitted"] == 1]


baseline_threshold = pit_laps["TyreLife"].mean()

print(f"Number of actual pit stops in the data: {len(pit_laps)}")
print(f"Average tyre age when pitting: {baseline_threshold:.1f} laps")
print(f"Median tyre age when pitting: {pit_laps['TyreLife'].median():.1f} laps")

Number of actual pit stops in the data: 325
Average tyre age when pitting: 20.0 laps
Median tyre age when pitting: 19.0 laps


In [15]:
# Builing the baseline rule

BASELINE_THRESHOLD = baseline_threshold #20.0

df["BaselinePrediction"] = (df["TyreLife"] >= BASELINE_THRESHOLD).astype(int)
print(df["BaselinePrediction"].value_counts())

BaselinePrediction
0    7996
1    3093
Name: count, dtype: int64


In [26]:
print("Baseline Rule Performance:")
print(f"Threshold: pit if TyreLife >= {BASELINE_THRESHOLD:.1f} laps\n\n")


print(classification_report(df["Pitted"], df["BaselinePrediction"]))

auc = roc_auc_score(df["Pitted"], df["BaselinePrediction"])
print(f"ROC- AUC: {auc:.4f}")

Baseline Rule Performance:
Threshold: pit if TyreLife >= 20.0 laps


              precision    recall  f1-score   support

           0       0.98      0.73      0.83     10764
           1       0.05      0.48      0.09       325

    accuracy                           0.72     11089
   macro avg       0.51      0.60      0.46     11089
weighted avg       0.95      0.72      0.81     11089

ROC- AUC: 0.6036


In [28]:
"""
Class 0 (didn't pit):
    precision: 0.98 --> when rule says "wont' pit" it is right 98% of the time 
    recall:    0.73 --> but it only catches 73% of all the real "didn't pit" cases



Class 1 (pitted):
    precision: 0.05 --> when rule says "pit" it is only right 5% of the time!
    recall:    0.48 --> but it only catxhes about half of the real pit stops


ROC-AUC : 0.6036 --> barely better than random guessing (0.5)
"""

'\n'

```

Reason for having a baseline :
if the XGBoost or whatever the model we chose performs worse than our baseline,
which means something is wrong potentially,
could be wrong features, data leakage, bad target variable, or bugs in the code.

so the baseline is a sanity check, catching mistakes before.

```